# 04 — Enriquecimiento de adaptaciones de videojuegos

Este notebook toma la lista semilla curada, consulta TMDB y OMDb para cada título y produce un CSV comparable. Las claves se leen del archivo local `.env` y no se guardan en el repositorio.

Flujo: `lista semilla → búsquedas API → JSON crudo fechado → CSV procesado`.

In [1]:
from datetime import datetime, timezone
from pathlib import Path
import json
import os
import re
import time

import pandas as pd
import requests
from dotenv import load_dotenv


def encontrar_raiz_proyecto(inicio: Path) -> Path:
    for carpeta in [inicio, *inicio.parents]:
        if (carpeta / '.env.example').exists() and (carpeta / 'data').exists():
            return carpeta
    raise FileNotFoundError('No se encontró la raíz del proyecto.')


RAIZ_PROYECTO = encontrar_raiz_proyecto(Path.cwd().resolve())
load_dotenv(RAIZ_PROYECTO / '.env', override=True)

TMDB_API_KEY = os.getenv('TMDB_API_KEY')
OMDB_API_KEY = os.getenv('OMDB_API_KEY')
if not TMDB_API_KEY or not OMDB_API_KEY:
    raise ValueError('Falta TMDB_API_KEY u OMDB_API_KEY en el archivo local .env.')

ruta_semilla = RAIZ_PROYECTO / 'data' / 'reference' / 'adaptaciones_videojuegos_semilla_2016_2025.csv'
semilla = pd.read_csv(ruta_semilla)
print(f'Películas semilla cargadas: {len(semilla)}')
semilla.head(3)

Películas semilla cargadas: 27


,id_semilla,titulo,anio,franquicia,tipo_adaptacion,criterio_inclusion,titulo_busqueda_tmdb,fuente_verificacion,estado_revision
0,vg_001,Warcraft,2016,Warcraft,pelicula_largometraje,Basada en la franquicia de videojuegos Warcraft,Warcraft,https://en.wikipedia.org/wiki/List_of_films_ba...,pendiente_enriquecimiento_api
1,vg_002,Ratchet & Clank,2016,Ratchet & Clank,pelicula_largometraje,Basada en el videojuego Ratchet & Clank,Ratchet & Clank,https://en.wikipedia.org/wiki/List_of_films_ba...,pendiente_enriquecimiento_api
2,vg_003,The Angry Birds Movie,2016,Angry Birds,pelicula_largometraje,Basada en el videojuego Angry Birds,The Angry Birds Movie,https://en.wikipedia.org/wiki/List_of_films_ba...,pendiente_enriquecimiento_api


In [2]:
BASE_TMDB = 'https://api.themoviedb.org/3'
URL_OMDB = 'https://www.omdbapi.com/'
sesion = requests.Session()


def consultar_tmdb(endpoint: str, **params) -> dict:
    respuesta = sesion.get(
        f'{BASE_TMDB}{endpoint}',
        params={'api_key': TMDB_API_KEY, **params},
        timeout=30,
    )
    respuesta.raise_for_status()
    return respuesta.json()


def elegir_resultado_tmdb(resultados: list, anio: int) -> tuple[dict | None, str]:
    candidatos_anio = [
        resultado for resultado in resultados
        if str(resultado.get('release_date', ''))[:4] == str(anio)
    ]
    if candidatos_anio:
        return candidatos_anio[0], 'coincidencia_anio'
    if resultados:
        return resultados[0], 'sin_coincidencia_anio'
    return None, 'sin_resultado'


def consultar_omdb(imdb_id: str | None, titulo: str, anio: int) -> dict:
    parametros = {'apikey': OMDB_API_KEY, 'type': 'movie', 'plot': 'short'}
    if imdb_id:
        parametros['i'] = imdb_id
    else:
        parametros.update({'t': titulo, 'y': anio})
    respuesta = sesion.get(URL_OMDB, params=parametros, timeout=30)
    respuesta.raise_for_status()
    return respuesta.json()


In [3]:
registros_tmdb = []
registros_omdb = []

for _, pelicula in semilla.iterrows():
    busqueda = consultar_tmdb(
        '/search/movie',
        query=pelicula['titulo_busqueda_tmdb'],
        year=int(pelicula['anio']),
        language='es-AR',
        include_adult='false',
    )
    seleccion, estado_busqueda = elegir_resultado_tmdb(busqueda.get('results', []), int(pelicula['anio']))
    detalle = consultar_tmdb(f"/movie/{seleccion['id']}", language='es-AR') if seleccion else None
    imdb_id = detalle.get('imdb_id') if detalle else None

    registros_tmdb.append({
        'id_semilla': pelicula['id_semilla'],
        'titulo_semilla': pelicula['titulo'],
        'anio_semilla': int(pelicula['anio']),
        'estado_busqueda_tmdb': estado_busqueda,
        'busqueda': busqueda,
        'detalle': detalle,
    })

    omdb = consultar_omdb(imdb_id, pelicula['titulo'], int(pelicula['anio'])) if detalle else {'Response': 'False', 'Error': 'Sin resultado TMDB'}
    registros_omdb.append({'id_semilla': pelicula['id_semilla'], 'respuesta': omdb})
    print(f"{pelicula['titulo']} ({pelicula['anio']}): {estado_busqueda}")
    time.sleep(0.1)

print(f'Consultas TMDB procesadas: {len(registros_tmdb)}')

Warcraft (2016): coincidencia_anio


Ratchet & Clank (2016): coincidencia_anio


The Angry Birds Movie (2016): coincidencia_anio


Assassin's Creed (2016): coincidencia_anio


Resident Evil: The Final Chapter (2016): coincidencia_anio


Resident Evil: Vendetta (2017): coincidencia_anio


... (17 outputs left out)

Sonic the Hedgehog 3 (2024): coincidencia_anio


A Minecraft Movie (2025): coincidencia_anio


Until Dawn (2025): coincidencia_anio


Five Nights at Freddy's 2 (2025): coincidencia_anio
Consultas TMDB procesadas: 27


In [4]:
def extraer_rating(ratings: list, fuente: str) -> str | None:
    for rating in ratings or []:
        if rating.get('Source') == fuente:
            return rating.get('Value')
    return None


def valor_financiero_o_nulo(valor: int | float | None) -> int | float | None:
    """TMDB devuelve 0 cuando no informa un monto; lo tratamos como faltante."""
    return valor if valor not in (None, 0) else None


omdb_por_semilla = {registro['id_semilla']: registro['respuesta'] for registro in registros_omdb}
filas = []

for _, pelicula in semilla.iterrows():
    registro_tmdb = next(registro for registro in registros_tmdb if registro['id_semilla'] == pelicula['id_semilla'])
    detalle = registro_tmdb['detalle'] or {}
    detalle_omdb = omdb_por_semilla[pelicula['id_semilla']]
    votos_imdb = detalle_omdb.get('imdbVotes', '')

    filas.append({
        **pelicula.to_dict(),
        'estado_busqueda_tmdb': registro_tmdb['estado_busqueda_tmdb'],
        'tmdb_id': detalle.get('id'),
        'imdb_id': detalle.get('imdb_id'),
        'titulo_tmdb': detalle.get('title'),
        'fecha_estreno': detalle.get('release_date'),
        'presupuesto_usd': valor_financiero_o_nulo(detalle.get('budget')),
        'recaudacion_usd': valor_financiero_o_nulo(detalle.get('revenue')),
        'generos_tmdb': ', '.join(genero['name'] for genero in detalle.get('genres', [])),
        'rating_tmdb': detalle.get('vote_average'),
        'votos_tmdb': detalle.get('vote_count'),
        'rating_imdb': detalle_omdb.get('imdbRating') if detalle_omdb.get('Response') == 'True' else None,
        'votos_imdb': int(re.sub(r'[^0-9]', '', votos_imdb)) if votos_imdb else None,
        'metascore': detalle_omdb.get('Metascore') if detalle_omdb.get('Response') == 'True' else None,
        'rating_rotten_tomatoes': extraer_rating(detalle_omdb.get('Ratings', []), 'Rotten Tomatoes'),
        'taquilla_omdb': detalle_omdb.get('BoxOffice') if detalle_omdb.get('Response') == 'True' else None,
        'estado_omdb': detalle_omdb.get('Response'),
    })

comparables = pd.DataFrame(filas)
fecha_extraccion = datetime.now(timezone.utc).date().isoformat()

def guardar_json(datos: object, ruta: Path) -> None:
    ruta.parent.mkdir(parents=True, exist_ok=True)
    with ruta.open('w', encoding='utf-8') as archivo:
        json.dump(datos, archivo, ensure_ascii=False, indent=2)

ruta_tmdb = RAIZ_PROYECTO / 'data' / 'raw' / 'tmdb' / f'adaptaciones_videojuegos_{fecha_extraccion}.json'
ruta_omdb = RAIZ_PROYECTO / 'data' / 'raw' / 'omdb' / f'adaptaciones_videojuegos_{fecha_extraccion}.json'
ruta_metadata = RAIZ_PROYECTO / 'data' / 'raw' / f'metadata_adaptaciones_videojuegos_{fecha_extraccion}.json'
ruta_csv = RAIZ_PROYECTO / 'data' / 'processed' / 'adaptaciones_videojuegos_2016_2025.csv'

guardar_json(registros_tmdb, ruta_tmdb)
guardar_json(registros_omdb, ruta_omdb)
guardar_json({
    'consulted_at_utc': datetime.now(timezone.utc).isoformat(),
    'input_file': str(ruta_semilla.relative_to(RAIZ_PROYECTO)),
    'records_requested': len(semilla),
    'tmdb_matches_by_status': pd.Series([registro['estado_busqueda_tmdb'] for registro in registros_tmdb]).value_counts().to_dict(),
    'omdb_successful_records': sum(registro['respuesta'].get('Response') == 'True' for registro in registros_omdb),
    'sources': ['TMDB /search/movie y /movie/{id}', 'OMDb por imdb_id o título y año'],
    'missing_value_policy': 'Los valores 0 de presupuesto_usd y recaudacion_usd de TMDB se convierten a nulos en el CSV procesado.',
}, ruta_metadata)

ruta_csv.parent.mkdir(parents=True, exist_ok=True)
comparables.to_csv(ruta_csv, index=False, encoding='utf-8')

print(f'CSV creado: {ruta_csv.relative_to(RAIZ_PROYECTO)}')
print(f'Filas: {len(comparables)} | IDs TMDB únicos: {comparables.tmdb_id.nunique(dropna=True)}')
print(comparables['estado_busqueda_tmdb'].value_counts(dropna=False))
comparables[['titulo', 'anio', 'estado_busqueda_tmdb', 'tmdb_id', 'rating_tmdb', 'rating_imdb']].head()

CSV creado: data\processed\adaptaciones_videojuegos_2016_2025.csv
Filas: 27 | IDs TMDB únicos: 27
estado_busqueda_tmdb
coincidencia_anio    27
Name: count, dtype: int64


,titulo,anio,estado_busqueda_tmdb,tmdb_id,rating_tmdb,rating_imdb
0,Warcraft,2016,coincidencia_anio,68735,6.428,6.7
1,Ratchet & Clank,2016,coincidencia_anio,234004,5.773,5.5
2,The Angry Birds Movie,2016,coincidencia_anio,153518,6.294,6.3
3,Assassin's Creed,2016,coincidencia_anio,121856,5.448,5.6
4,Resident Evil: The Final Chapter,2016,coincidencia_anio,173897,6.102,5.5
